In [2]:
import requests

# ---------------- EXTRACT ----------------

url = "https://api.frankfurter.dev/v2/rates"      

#latest rates -- acoording to documentation
params = {
    "base": "EUR",
    "quotes": "USD,GBP,CHF,CNY"
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()

data = response.json()

print(data)


import pandas as pd

# ---------------- TRANSFORM ----------------

df = pd.DataFrame(data)

df["date"] = pd.to_datetime(df["date"])

display(df)
print(df.dtypes)

# ---------------- VALIDATE ----------------

if df.isna().sum().sum() != 0:
    raise ValueError("Validation failed: missing values found")

if (df["rate"] <= 0).any():
    raise ValueError("Validation failed: invalid exchange rate")

if df.duplicated(subset=["date", "base", "quote"]).any():
    raise ValueError("Validation failed: duplicate exchange-rate records")

print("Validation passed")

# ---------------- PANDAS → SPARK ----------------

spark_df = spark.createDataFrame(df)

display(spark_df)
spark_df.printSchema()

spark_df.createOrReplaceTempView("incoming_rates")


# ---------------- MERGE INTO DELTA TABLE ----------------

merge_result = spark.sql("""
MERGE INTO exchange_rates AS target
USING incoming_rates AS source

ON  target.date = source.date
AND target.base = source.base
AND target.quote = source.quote

WHEN MATCHED THEN
    UPDATE SET
        target.rate = source.rate

WHEN NOT MATCHED THEN
    INSERT (
        date,
        base,
        quote,
        rate
    )
    VALUES (
        source.date,
        source.base,
        source.quote,
        source.rate
    )
""")

display(merge_result)


# ---------------- VERIFY LOAD ----------------

print("Total rows:", spark.table("exchange_rates").count())

spark.sql("""
SELECT *
FROM exchange_rates
WHERE date = CURRENT_DATE()
ORDER BY quote
""").show()

StatementMeta(, 0163d015-c83d-44cc-bcb9-d7ab36127ef3, 4, Finished, Available, Finished, False)

[{'date': '2026-10-01', 'base': 'EUR', 'quote': 'CHF', 'rate': 0.94639}, {'date': '2026-10-01', 'base': 'EUR', 'quote': 'CNY', 'rate': 7.6053}, {'date': '2026-10-01', 'base': 'EUR', 'quote': 'GBP', 'rate': 0.85574}, {'date': '2026-10-01', 'base': 'EUR', 'quote': 'USD', 'rate': 1.1352}]


SynapseWidget(Synapse.DataFrame, 4f77858c-8ba1-4139-9db9-69ac90358255)

date     datetime64[ns]
base             object
quote            object
rate            float64
dtype: object
Validation passed


SynapseWidget(Synapse.DataFrame, ff98170f-eadc-49aa-b8e6-b597b15f16b2)

root
 |-- date: timestamp (nullable = true)
 |-- base: string (nullable = true)
 |-- quote: string (nullable = true)
 |-- rate: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 61f24f4f-3d38-4178-b732-4a6e0e9a5312)

Total rows: 3013
+-------------------+----+-----+-------+
|               date|base|quote|   rate|
+-------------------+----+-----+-------+
|2026-10-01 00:00:00| EUR|  CHF|0.94639|
|2026-10-01 00:00:00| EUR|  CNY| 7.6053|
|2026-10-01 00:00:00| EUR|  GBP|0.85574|
|2026-10-01 00:00:00| EUR|  USD| 1.1352|
+-------------------+----+-----+-------+

